# MoTrPAC rat-heart transcriptomic WGCNA — Step 4

This notebook constructs a signed heart RNA co-expression network using the 5,000 most variable normalized transcript features. It identifies modules and hub genes, correlates module eigengenes with sex, training duration, VO₂max/body-composition traits where available, and tests the same 13 sex/time contrasts at module level.

Outputs are saved in `My Drive/MoTrPAC_2day_hackathon/results/04_rat_heart_transcriptomic_WGCNA/`.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/MoTrPAC_2day_hackathon')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('Project folder:', PROJECT_DIR)


In [ ]:
import shutil, subprocess
if shutil.which('R') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'r-base', 'r-base-dev', 'git', 'build-essential', 'libcurl4-openssl-dev', 'libssl-dev', 'libxml2-dev', 'libfontconfig1-dev', 'libharfbuzz-dev', 'libfribidi-dev', 'libfreetype6-dev', 'libpng-dev', 'libtiff5-dev', 'libjpeg-dev', 'libopenblas-dev', 'liblapack-dev', 'gfortran'], check=True)
!R --version


## Create the R WGCNA script


In [ ]:
r_code = r'''
# ============================================================
# MoTrPAC Rat Heart Hackathon
# Step 4: Transcriptomic WGCNA and module-trait associations
# Author: Upama
# ============================================================

options(timeout = 1e5)
options(repos = c(CRAN = "https://cloud.r-project.org"))
set.seed(42)

message("R version: ", R.version.string)
message("Working directory: ", getwd())

# 1. Install required packages
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")

bioc_packages <- c("impute", "preprocessCore", "GO.db", "AnnotationDbi", "limma")
missing_bioc <- bioc_packages[!vapply(bioc_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_bioc) > 0) {
  BiocManager::install(missing_bioc, ask = FALSE, update = FALSE, Ncpus = 2)
}

cran_packages <- c(
  "remotes", "WGCNA", "dynamicTreeCut", "fastcluster",
  "dplyr", "tidyr", "tibble", "readr", "ggplot2",
  "stringr", "forcats", "scales"
)
missing_cran <- cran_packages[!vapply(cran_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_cran) > 0) {
  install.packages(missing_cran, dependencies = c("Depends", "Imports", "LinkingTo"), Ncpus = 2)
}

if (!requireNamespace("MotrpacRatTraining6moData", quietly = TRUE)) {
  remotes::install_github(
    "MoTrPAC/MotrpacRatTraining6moData",
    upgrade = "never", dependencies = FALSE,
    build_vignettes = FALSE, force = TRUE
  )
}

suppressPackageStartupMessages({
  library(MotrpacRatTraining6moData)
  library(WGCNA)
  library(limma)
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(readr)
  library(ggplot2)
  library(stringr)
  library(forcats)
})
allowWGCNAThreads(nThreads = 2)

# 2. Load data
data("PHENO", package = "MotrpacRatTraining6moData")
data("OUTLIERS", package = "MotrpacRatTraining6moData")
data("TRNSCRPT_META", package = "MotrpacRatTraining6moData")
data("TRNSCRPT_HEART_NORM_DATA", package = "MotrpacRatTraining6moData")
data("FEATURE_TO_GENE", package = "MotrpacRatTraining6moData")

norm_data <- TRNSCRPT_HEART_NORM_DATA
output_dir <- file.path("results", "04_rat_heart_transcriptomic_WGCNA")
plot_dir <- file.path(output_dir, "plots")
module_dir <- file.path(output_dir, "module_tables")
dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(module_dir, recursive = TRUE, showWarnings = FALSE)

# 3. Sample metadata
pheno_ids <- unique(c(
  rownames(PHENO),
  if ("viallabel" %in% colnames(PHENO)) as.character(PHENO$viallabel) else character()
))
sample_ids <- intersect(colnames(norm_data), pheno_ids)
if (length(sample_ids) == 0) stop("No normalized heart RNA-seq sample columns matched PHENO.")

if ("viallabel" %in% colnames(PHENO)) {
  sample_metadata <- PHENO |>
    filter(as.character(viallabel) %in% sample_ids) |>
    distinct(viallabel, .keep_all = TRUE)
  sample_metadata <- sample_metadata[match(sample_ids, as.character(sample_metadata$viallabel)), , drop = FALSE]
} else {
  sample_metadata <- PHENO[sample_ids, , drop = FALSE]
  sample_metadata$viallabel <- rownames(sample_metadata)
}

sample_metadata <- sample_metadata |>
  mutate(
    viallabel = as.character(viallabel),
    sex = factor(str_to_title(as.character(sex)), levels = c("Female", "Male")),
    timepoint = ifelse(as.character(group) == "control", "SED", toupper(as.character(group))),
    timepoint = factor(timepoint, levels = c("SED", "1W", "2W", "4W", "8W")),
    training_week = as.numeric(recode(as.character(timepoint), "SED" = 0, "1W" = 1, "2W" = 2, "4W" = 4, "8W" = 8)),
    trained = as.numeric(timepoint != "SED"),
    sex_male = as.numeric(sex == "Male"),
    sex_x_training = sex_male * training_week,
    exp_group = interaction(substr(as.character(sex), 1, 1), timepoint, sep = "_"),
    exp_group = factor(exp_group, levels = c(
      "F_SED", "F_1W", "F_2W", "F_4W", "F_8W",
      "M_SED", "M_1W", "M_2W", "M_4W", "M_8W"
    ))
  )

official_outliers <- intersect(as.character(OUTLIERS$viallabel), sample_metadata$viallabel)
sample_metadata <- sample_metadata |> filter(!viallabel %in% official_outliers)
sample_ids <- sample_metadata$viallabel
rownames(sample_metadata) <- sample_ids

# 4. Expression matrix
if (!"feature_ID" %in% colnames(norm_data)) stop("Normalized data do not contain feature_ID.")
expr_matrix <- norm_data |>
  select(feature_ID, all_of(sample_ids)) |>
  column_to_rownames("feature_ID") |>
  as.matrix()
storage.mode(expr_matrix) <- "numeric"

valid_features <- apply(expr_matrix, 1, function(x) all(is.finite(x)) && !is.na(sd(x)) && sd(x) > 0)
expr_matrix <- expr_matrix[valid_features, , drop = FALSE]
feature_variance <- apply(expr_matrix, 1, var)
n_wgcna_features <- min(5000L, length(feature_variance))
selected_features <- names(sort(feature_variance, decreasing = TRUE))[seq_len(n_wgcna_features)]
expr_matrix <- expr_matrix[selected_features, , drop = FALSE]

datExpr <- t(expr_matrix)
rownames(datExpr) <- sample_ids

gsg <- goodSamplesGenes(datExpr, verbose = 3)
if (!gsg$allOK) {
  datExpr <- datExpr[gsg$goodSamples, gsg$goodGenes, drop = FALSE]
  sample_metadata <- sample_metadata[rownames(datExpr), , drop = FALSE]
}

# 5. Feature annotations
feature_annotation <- FEATURE_TO_GENE |>
  filter(feature_ID %in% colnames(datExpr)) |>
  select(feature_ID, gene_symbol, entrez_gene) |>
  distinct() |>
  group_by(feature_ID) |>
  summarise(
    gene_symbol = ifelse(all(is.na(gene_symbol)), NA_character_, paste(unique(na.omit(gene_symbol)), collapse = ";")),
    entrez_gene = ifelse(all(is.na(entrez_gene)), NA_character_, paste(unique(na.omit(as.character(entrez_gene))), collapse = ";")),
    .groups = "drop"
  )

# 6. Sample clustering
sample_tree <- hclust(dist(datExpr), method = "average")
png(file.path(plot_dir, "WGCNA_sample_clustering.png"), width = 1800, height = 1100, res = 220)
plot(sample_tree, main = "Rat-heart RNA-seq sample clustering for WGCNA", xlab = "", sub = "", cex = 0.75)
dev.off()

# 7. Soft-threshold selection
powers <- c(1:10, seq(12, 20, by = 2))
sft <- pickSoftThreshold(
  datExpr, powerVector = powers, networkType = "signed",
  corFnc = "bicor", corOptions = list(maxPOutliers = 0.1),
  verbose = 5, blockSize = 2000
)
fit_indices <- as.data.frame(sft$fitIndices)
candidate_rows <- which(fit_indices$SFT.R.sq >= 0.80 & fit_indices$slope < 0 & fit_indices$mean.k. > 1)
if (length(candidate_rows) > 0) {
  soft_power <- fit_indices$Power[min(candidate_rows)]
} else {
  eligible <- which(fit_indices$mean.k. > 1)
  soft_power <- if (length(eligible) == 0) 6 else fit_indices$Power[eligible[which.max(fit_indices$SFT.R.sq[eligible])]]
}
message("Selected soft-thresholding power: ", soft_power)

png(file.path(plot_dir, "soft_threshold_selection.png"), width = 2000, height = 1000, res = 180)
par(mfrow = c(1, 2))
plot(fit_indices$Power, fit_indices$SFT.R.sq, xlab = "Soft-thresholding power", ylab = "Scale-free topology fit (R²)", type = "n", main = "Scale independence")
text(fit_indices$Power, fit_indices$SFT.R.sq, labels = fit_indices$Power, col = "red")
abline(h = 0.80, col = "red", lty = 2); abline(v = soft_power, col = "blue", lty = 2)
plot(fit_indices$Power, fit_indices$mean.k., xlab = "Soft-thresholding power", ylab = "Mean connectivity", type = "n", main = "Mean connectivity")
text(fit_indices$Power, fit_indices$mean.k., labels = fit_indices$Power, col = "red")
abline(v = soft_power, col = "blue", lty = 2)
dev.off()

# 8. Network construction
net <- blockwiseModules(
  datExpr,
  power = soft_power,
  networkType = "signed",
  TOMType = "signed",
  corType = "bicor",
  maxPOutliers = 0.1,
  maxBlockSize = 6000,
  minModuleSize = 30,
  deepSplit = 2,
  mergeCutHeight = 0.25,
  pamRespectsDendro = FALSE,
  numericLabels = TRUE,
  saveTOMs = FALSE,
  randomSeed = 42,
  nThreads = 2,
  verbose = 4
)

module_colors <- labels2colors(net$colors)
names(module_colors) <- colnames(datExpr)
MEs <- moduleEigengenes(datExpr, colors = module_colors)$eigengenes
MEs <- orderMEs(MEs)
rownames(MEs) <- rownames(datExpr)

for (block in seq_along(net$dendrograms)) {
  block_features <- net$blockGenes[[block]]
  png(file.path(plot_dir, paste0("WGCNA_feature_dendrogram_block_", block, ".png")), width = 2400, height = 1200, res = 180)
  plotDendroAndColors(
    net$dendrograms[[block]], module_colors[block_features], "Module colors",
    dendroLabels = FALSE, hang = 0.03, addGuide = TRUE, guideHang = 0.05,
    main = paste("Heart transcriptomic modules — block", block)
  )
  dev.off()
}

# 9. Module membership and hub genes
kME <- signedKME(datExpr, MEs, outputColumnName = "kME")
get_own_kme <- function(feature_id, module_color) {
  column_name <- paste0("kME", module_color)
  if (!column_name %in% colnames(kME)) return(NA_real_)
  as.numeric(kME[feature_id, column_name])
}

module_assignment <- tibble(
  feature_ID = colnames(datExpr),
  module = unname(module_colors[colnames(datExpr)])
) |>
  rowwise() |>
  mutate(module_membership = get_own_kme(feature_ID, module)) |>
  ungroup() |>
  left_join(feature_annotation, by = "feature_ID")

module_sizes <- module_assignment |> count(module, name = "n_features") |> arrange(desc(n_features))
hub_genes <- module_assignment |>
  filter(module != "grey") |>
  group_by(module) |>
  arrange(desc(abs(module_membership)), .by_group = TRUE) |>
  slice_head(n = 10) |>
  ungroup()

# 10. Animal phenotypes
first_numeric <- function(x) {
  y <- suppressWarnings(as.numeric(as.character(x)))
  y <- y[is.finite(y)]
  if (length(y) == 0) NA_real_ else y[1]
}

trait_candidates <- list(
  vo2max = c("vo2_max", "vo2.max.test.vo2_max", "vo2_max_test___vo2_max"),
  vo2max_change = c("calculated.variables.vo2_max_change", "calculated_variables___vo2_max_change"),
  pct_body_fat_change = c("calculated.variables.pct_body_fat_change", "calculated_variables___pct_body_fat_change"),
  pct_body_lean_change = c("calculated.variables.pct_body_lean_change", "calculated_variables___pct_body_lean_change"),
  terminal_body_weight = c("terminal.weight.bw", "terminal_weight___bw"),
  nmr_fat = c("nmr.testing.nmr_fat", "nmr_testing___nmr_fat")
)

available_traits <- list()
for (trait_name in names(trait_candidates)) {
  available <- trait_candidates[[trait_name]][trait_candidates[[trait_name]] %in% colnames(PHENO)]
  if (length(available) > 0) {
    candidate_column <- available[1]
    pid_trait <- PHENO |>
      filter(pid %in% sample_metadata$pid) |>
      group_by(pid) |>
      summarise(value = first_numeric(.data[[candidate_column]]), .groups = "drop")
    values <- pid_trait$value[match(sample_metadata$pid, pid_trait$pid)]
    if (sum(is.finite(values)) >= 8 && sd(values, na.rm = TRUE) > 0) available_traits[[trait_name]] <- values
  }
}

trait_data <- data.frame(
  sex_male = sample_metadata$sex_male,
  training_week = sample_metadata$training_week,
  trained = sample_metadata$trained,
  sex_x_training = sample_metadata$sex_x_training,
  row.names = sample_metadata$viallabel
)
for (trait_name in names(available_traits)) trait_data[[trait_name]] <- available_traits[[trait_name]]

# 11. Module-trait Spearman correlations
correlation_rows <- list(); row_index <- 1
for (module_name in colnames(MEs)) {
  for (trait_name in colnames(trait_data)) {
    x <- MEs[, module_name]; y <- trait_data[, trait_name]
    complete <- is.finite(x) & is.finite(y); n_complete <- sum(complete)
    if (n_complete >= 8 && sd(x[complete]) > 0 && sd(y[complete]) > 0) {
      test <- suppressWarnings(cor.test(x[complete], y[complete], method = "spearman", exact = FALSE))
      correlation_rows[[row_index]] <- tibble(
        module = sub("^ME", "", module_name), module_eigengene = module_name,
        trait = trait_name, rho = unname(test$estimate), p_value = test$p.value, n = n_complete
      )
      row_index <- row_index + 1
    }
  }
}

module_trait_results <- bind_rows(correlation_rows) |>
  mutate(FDR = p.adjust(p_value, method = "BH"), significant_FDR05 = FDR < 0.05) |>
  left_join(module_sizes, by = "module")

module_trait_plot_data <- module_trait_results |>
  mutate(
    module_label = paste0(module, " (n=", n_features, ")"),
    significance = case_when(FDR < 0.001 ~ "***", FDR < 0.01 ~ "**", FDR < 0.05 ~ "*", TRUE ~ "")
  )
module_order <- module_trait_plot_data |>
  group_by(module_label) |>
  summarise(max_abs_rho = max(abs(rho), na.rm = TRUE), .groups = "drop") |>
  arrange(desc(max_abs_rho)) |>
  pull(module_label)
module_trait_plot_data <- module_trait_plot_data |>
  mutate(module_label = factor(module_label, levels = rev(module_order)))

module_trait_heatmap <- ggplot(module_trait_plot_data, aes(x = trait, y = module_label, fill = rho)) +
  geom_tile(colour = "white") +
  geom_text(aes(label = significance), size = 4) +
  scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B", midpoint = 0, limits = c(-1, 1)) +
  labs(
    title = "Rat-heart transcriptomic module–trait associations",
    subtitle = "Spearman correlations; * FDR<0.05, ** FDR<0.01, *** FDR<0.001",
    x = NULL, y = "WGCNA module", fill = "Spearman\\nrho"
  ) +
  theme_bw(base_size = 10) +
  theme(plot.title = element_text(face = "bold"), axis.text.x = element_text(angle = 40, hjust = 1), panel.grid = element_blank())

ggsave(
  file.path(plot_dir, "module_trait_correlation_heatmap.png"),
  module_trait_heatmap, width = 11,
  height = max(6, 0.35 * length(unique(module_trait_plot_data$module))),
  dpi = 300, limitsize = FALSE
)

# 12. Module eigengene profiles
ME_metadata <- as.data.frame(MEs) |>
  rownames_to_column("viallabel") |>
  left_join(sample_metadata |> select(viallabel, sex, timepoint, exp_group), by = "viallabel")

ME_group_summary <- ME_metadata |>
  pivot_longer(cols = starts_with("ME"), names_to = "module_eigengene", values_to = "eigengene") |>
  group_by(module_eigengene, sex, timepoint, exp_group) |>
  summarise(mean_eigengene = mean(eigengene), SE = sd(eigengene) / sqrt(n()), n = n(), .groups = "drop") |>
  group_by(module_eigengene) |>
  mutate(scaled_mean = as.numeric(scale(mean_eigengene))) |>
  ungroup() |>
  mutate(module = sub("^ME", "", module_eigengene))

ME_profile_heatmap <- ggplot(ME_group_summary, aes(x = exp_group, y = fct_rev(module), fill = scaled_mean)) +
  geom_tile(colour = "white") +
  scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B", midpoint = 0) +
  labs(title = "Module eigengene profiles across sex and training time", x = NULL, y = "WGCNA module", fill = "Scaled\\nmean") +
  theme_bw(base_size = 10) +
  theme(plot.title = element_text(face = "bold"), axis.text.x = element_text(angle = 45, hjust = 1), panel.grid = element_blank())

ggsave(
  file.path(plot_dir, "module_eigengene_group_heatmap.png"),
  ME_profile_heatmap, width = 11, height = max(6, 0.30 * ncol(MEs)), dpi = 300, limitsize = FALSE
)

# 13. Formal module-level contrasts
qc_columns <- c("viallabel", "RIN", "pct_globin", "pct_umi_dup", "median_5_3_bias")
if (all(qc_columns %in% colnames(TRNSCRPT_META))) {
  qc_data <- TRNSCRPT_META |>
    transmute(
      viallabel = as.character(viallabel), rin = as.numeric(RIN),
      pct_globin = as.numeric(pct_globin), pct_umi_dup = as.numeric(pct_umi_dup),
      median_5_3_bias = as.numeric(median_5_3_bias)
    ) |>
    distinct(viallabel, .keep_all = TRUE)
  sample_metadata <- sample_metadata |> left_join(qc_data, by = "viallabel")
  fix_covariate <- function(x) {
    x[is.na(x)] <- mean(x, na.rm = TRUE)
    if (sd(x) == 0) rep(0, length(x)) else as.numeric(scale(x))
  }
  for (nm in c("rin", "pct_globin", "pct_umi_dup", "median_5_3_bias")) sample_metadata[[nm]] <- fix_covariate(sample_metadata[[nm]])
  design <- model.matrix(~ 0 + exp_group + rin + pct_globin + pct_umi_dup + median_5_3_bias, data = sample_metadata)
} else {
  design <- model.matrix(~ 0 + exp_group, data = sample_metadata)
}
colnames(design) <- sub("^exp_group", "", colnames(design))
rownames(design) <- sample_metadata$viallabel
ME_expression <- t(MEs[sample_metadata$viallabel, , drop = FALSE])
base_fit <- lmFit(ME_expression, design)

contrast_groups <- list(
  trained_vs_SED = c(
    "F_1W - F_SED", "F_2W - F_SED", "F_4W - F_SED", "F_8W - F_SED",
    "M_1W - M_SED", "M_2W - M_SED", "M_4W - M_SED", "M_8W - M_SED"
  ),
  MvF_SED = c("M_SED - F_SED"),
  MvF_exercise_response = c(
    "(M_1W - M_SED) - (F_1W - F_SED)",
    "(M_2W - M_SED) - (F_2W - F_SED)",
    "(M_4W - M_SED) - (F_4W - F_SED)",
    "(M_8W - M_SED) - (F_8W - F_SED)"
  )
)

run_module_contrasts <- function(group_name, contrast_vector) {
  cm <- makeContrasts(contrasts = contrast_vector, levels = design)
  fit <- eBayes(contrasts.fit(base_fit, cm), robust = TRUE)
  out <- bind_rows(lapply(seq_along(contrast_vector), function(i) {
    tab <- topTable(fit, coef = i, number = Inf, sort.by = "none", adjust.method = "BH")
    tab$module_eigengene <- rownames(tab)
    tab$module <- sub("^ME", "", tab$module_eigengene)
    tab$contrast <- contrast_vector[i]
    tab$contrast_group <- group_name
    tab
  }))
  out$global_FDR <- p.adjust(out$P.Value, method = "BH")
  out |> mutate(significant_global_FDR05 = global_FDR < 0.05)
}

module_contrast_results <- bind_rows(lapply(names(contrast_groups), function(group_name) {
  run_module_contrasts(group_name, contrast_groups[[group_name]])
}))

module_contrast_counts <- module_contrast_results |>
  group_by(contrast_group, contrast) |>
  summarise(
    tested_modules = n(),
    significant_modules = sum(significant_global_FDR05, na.rm = TRUE),
    positive_modules = sum(significant_global_FDR05 & logFC > 0, na.rm = TRUE),
    negative_modules = sum(significant_global_FDR05 & logFC < 0, na.rm = TRUE),
    minimum_global_FDR = min(global_FDR, na.rm = TRUE),
    .groups = "drop"
  )

# 14. Save outputs
analysis_inventory <- tibble(
  item = c(
    "RNA-seq samples", "Normalized features available", "Features used for WGCNA",
    "Network type", "Correlation", "Selected soft-threshold power",
    "Minimum module size", "Module merge cut height", "Detected modules excluding grey",
    "Official outliers removed", "WGCNA version", "Data package version"
  ),
  value = c(
    nrow(datExpr), nrow(norm_data), ncol(datExpr), "signed", "biweight midcorrelation",
    soft_power, 30, 0.25, sum(module_sizes$module != "grey"), length(official_outliers),
    as.character(packageVersion("WGCNA")), as.character(packageVersion("MotrpacRatTraining6moData"))
  )
)

write_csv(analysis_inventory, file.path(output_dir, "WGCNA_analysis_inventory.csv"))
write_csv(fit_indices, file.path(output_dir, "soft_threshold_fit_indices.csv"))
write_csv(as.data.frame(trait_data) |> rownames_to_column("viallabel"), file.path(output_dir, "WGCNA_trait_data.csv"))
write_csv(as.data.frame(MEs) |> rownames_to_column("viallabel"), file.path(output_dir, "module_eigengenes.csv"))
write_csv(module_sizes, file.path(output_dir, "module_sizes.csv"))
write_csv(module_assignment, file.path(output_dir, "feature_module_assignments.csv"))
write_csv(hub_genes, file.path(output_dir, "top10_hub_genes_per_module.csv"))
write_csv(module_trait_results, file.path(output_dir, "module_trait_correlations.csv"))
write_csv(module_trait_results |> filter(significant_FDR05) |> arrange(FDR), file.path(output_dir, "significant_module_trait_correlations.csv"))
write_csv(ME_group_summary, file.path(output_dir, "module_eigengene_group_summary.csv"))
write_csv(module_contrast_results, file.path(output_dir, "module_contrast_results.csv"))
write_csv(module_contrast_counts, file.path(output_dir, "module_contrast_counts.csv"))

for (module_name in unique(module_assignment$module)) {
  write_csv(
    module_assignment |> filter(module == module_name) |> arrange(desc(abs(module_membership))),
    file.path(module_dir, paste0("module_", module_name, "_features.csv"))
  )
}

saveRDS(
  list(
    datExpr = datExpr, sample_metadata = sample_metadata, soft_threshold = sft,
    soft_power = soft_power, network = net, module_colors = module_colors,
    module_eigengenes = MEs, module_assignment = module_assignment,
    module_trait_results = module_trait_results,
    module_contrast_results = module_contrast_results
  ),
  file.path(output_dir, "rat_heart_WGCNA_objects.rds"), compress = "xz"
)

capture.output(sessionInfo(), file = file.path(output_dir, "sessionInfo.txt"))

cat("\\n================ WGCNA INVENTORY ================\\n")
print(analysis_inventory)
cat("\\n================ MODULE SIZES ===================\\n")
print(module_sizes)
cat("\\n================ SIGNIFICANT MODULE-TRAIT RESULTS ================\\n")
print(module_trait_results |> filter(significant_FDR05) |> arrange(FDR))
cat("\\n================ MODULE CONTRAST COUNTS ==========================\\n")
print(module_contrast_counts)
cat("\\nSUCCESS: WGCNA outputs saved to:\\n")
cat(normalizePath(output_dir, winslash = "/", mustWork = FALSE), "\\n")

'''
script_path = PROJECT_DIR / '04_rat_heart_transcriptomic_WGCNA.R'
script_path.write_text(r_code, encoding='utf-8')
print('Saved:', script_path)


## Run WGCNA

Package installation and network construction can take time. Keep the Colab tab open until the success message appears.


In [ ]:
import subprocess
log_path = PROJECT_DIR / '04_WGCNA_full.log'
process = subprocess.Popen(['Rscript', '--vanilla', '04_rat_heart_transcriptomic_WGCNA.R'], cwd=str(PROJECT_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with log_path.open('w', encoding='utf-8') as log_file:
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
return_code = process.wait()
print('\nR return code:', return_code)
print('Full log:', log_path)
if return_code != 0:
    lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
    print('\n===== LAST 180 LOG LINES =====')
    print('\n'.join(lines[-180:]))
    raise RuntimeError('WGCNA failed. Share the displayed final log lines.')
print('WGCNA completed successfully.')


## Review the main WGCNA outputs


In [ ]:
import pandas as pd
from IPython.display import Image, display
RESULTS_DIR = PROJECT_DIR / 'results' / '04_rat_heart_transcriptomic_WGCNA'
print('Results folder exists:', RESULTS_DIR.exists())
for filename in ['WGCNA_analysis_inventory.csv', 'module_sizes.csv', 'significant_module_trait_correlations.csv', 'module_contrast_counts.csv', 'top10_hub_genes_per_module.csv']:
    path = RESULTS_DIR / filename
    print(f'\n### {filename}')
    if path.exists():
        df = pd.read_csv(path)
        display(df.head(50))
        print('Rows:', len(df))
    else:
        print('File not found:', path)
for relative_path in ['plots/WGCNA_sample_clustering.png', 'plots/soft_threshold_selection.png', 'plots/WGCNA_feature_dendrogram_block_1.png', 'plots/module_trait_correlation_heatmap.png', 'plots/module_eigengene_group_heatmap.png']:
    path = RESULTS_DIR / relative_path
    print(f'\n### {relative_path}')
    if path.exists(): display(Image(filename=str(path)))
    else: print('Plot not found:', path)


In [ ]:
import shutil
archive_path = shutil.make_archive(str(PROJECT_DIR / 'rat_heart_WGCNA_results'), 'zip', root_dir=str(RESULTS_DIR))
print('Results ZIP saved in Drive:')
print(archive_path)
